# LeetCode 3420 — Count Non-Decreasing Subarrays After K Operations bằng Python

- Link: https://leetcode.com/problems/count-non-decreasing-subarrays-after-k-operations/
- Độ khó: **Hard**
- Chủ đề: **Chi phí = Σ (max tiền tố − phần tử)**, **hai con trỏ** (tính đơn điệu), **hàng đợi hai đầu chứa các khối max tiền tố** (ngăn xếp đơn điệu), **nhảy nhị phân** trên cây "phần tử lớn hơn kế tiếp"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countNonDecreasingSubarrays(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Chứng minh chi phí tối thiểu để làm mảng con `[l, r]` không giảm (chỉ được **tăng**) là `Σ_{j=l}^{r} (max(nums[l..j]) − nums[j])`.
2. Nhận ra tính **đơn điệu**: mảng con của mảng con khả thi cũng khả thi ⇒ hai con trỏ.
3. Duy trì chi phí khi **thêm bên trái** và **bỏ bên phải**: biểu diễn max tiền tố của cửa sổ thành **các khối** giá trị tăng dần trong một `deque`.
4. Cách thay thế: tính `cost(l, r)` trong `O(log n)` bằng **binary lifting** trên chuỗi "phần tử lớn hơn kế tiếp" + tổng tiền tố.
5. Đối chiếu với vét cạn `O(n²)`.

## 2. Đề bài & ví dụ

Với mỗi mảng con của `nums`, được tăng phần tử (mỗi lần +1) tối đa `k` lần (các mảng con độc lập). Đếm số mảng con có thể trở thành **không giảm**.

**Ví dụ 1:** `nums = [6,3,1,2,4,4], k = 7` → `17`. **Ví dụ 2:** `nums = [6,3,1,3,6], k = 4` → `12`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i], k ≤ 10^9`.

In [1]:
from typing import List

examples = [([6, 3, 1, 2, 4, 4], 7, 17), ([6, 3, 1, 3, 6], 4, 12)]
for nums, k, exp in examples:
    print(nums, k, '-> kỳ vọng', exp)

[6, 3, 1, 2, 4, 4] 7 -> kỳ vọng 17
[6, 3, 1, 3, 6] 4 -> kỳ vọng 12


## 3. Kiến thức nền

### 3.1. Chi phí tối thiểu để không giảm (chỉ tăng)

Chỉ được tăng ⇒ phần tử `j` phải ≥ mọi phần tử (đã tăng) trước nó, nên ít nhất là `max(nums[l..j])`. Đặt mỗi phần tử đúng bằng max tiền tố là đủ (dãy max tiền tố không giảm) ⇒ chi phí tối thiểu `Σ (prefmax − nums[j])`.

In [2]:
def cost_direct(a):
    mx, c = 0, 0
    for x in a:
        mx = max(mx, x); c += mx - x
    return c

print(cost_direct([6, 3, 1]), cost_direct([3, 1, 3, 6]), cost_direct([1, 2, 4, 4]))

8 2 0


### 3.2. Tính đơn điệu

Nếu `[l, r]` khả thi thì mọi `[l', r'] ⊆ [l, r]` khả thi (bỏ phần tử đầu chỉ làm max tiền tố nhỏ đi, bỏ phần tử cuối bớt một số hạng không âm). ⇒ với mỗi `l`, các `r` khả thi là một **đoạn** `[l, R(l)]`, và `R(l)` không giảm theo `l`.

### 3.3. Phần tử lớn hơn kế tiếp (next greater)

In [3]:
def next_greater(a):
    n, ng, st = len(a), [len(a)] * len(a), []
    for i in range(n):
        while st and a[st[-1]] < a[i]:
            ng[st.pop()] = i
        st.append(i)
    return ng

print(next_greater([6, 3, 1, 2, 4, 4]))

[6, 4, 3, 4, 6, 6]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1, `k = 7`. Với `l = 0` (`6` đứng đầu), max tiền tố luôn là 6: `[6,3]` tốn 3, `[6,3,1]` tốn `3 + 5 = 8 > 7` ⇒ `R(0) = 1` (2 mảng con). Với `l = 1`: `[3,1,2,4,4]` tốn `2 + 1 = 3` ⇒ `R(1) = 5` (5 mảng con). Tổng các `l`: `2 + 5 + 4 + 3 + 2 + 1 = 17` ✓.

In [4]:
nums, k = examples[0][:2]
for l in range(len(nums)):
    R = max(r for r in range(l, len(nums)) if cost_direct(nums[l:r + 1]) <= k)
    print(f"l = {l}: R = {R}, số mảng con = {R - l + 1}")

l = 0: R = 1, số mảng con = 2
l = 1: R = 5, số mảng con = 5
l = 2: R = 5, số mảng con = 4
l = 3: R = 5, số mảng con = 3
l = 4: R = 5, số mảng con = 2
l = 5: R = 5, số mảng con = 1


## 5. Cách 1 — Brute Force: mỗi `l`, mở rộng `r` tới khi vượt `k`

In [5]:
def count_brute(nums, k):
    n, c = len(nums), 0
    for l in range(n):
        mx = cost = 0
        for r in range(l, n):
            mx = max(mx, nums[r]); cost += mx - nums[r]
            if cost > k:
                break
            c += 1
    return c

for nums, k, exp in examples:
    print(count_brute(nums, k), '| kỳ vọng', exp)

17 | kỳ vọng 17
12 | kỳ vọng 12


**Độ phức tạp:** `O(n²)` tệ nhất (khi `k` lớn).

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt `l` **từ phải sang trái**, giữ cửa sổ `[l, r]` và chi phí của nó:

- **Thêm `nums[l]` vào bên trái:** max tiền tố của cửa sổ là dãy khối `(giá trị, số phần tử)` tăng dần từ trái sang. Phần tử mới `v` "nuốt" mọi khối đầu có giá trị `≤ v`: mỗi khối `(b, c)` làm chi phí tăng `(v − b) · c`. Đẩy khối `(v, tổng số)` vào đầu.
- **Bỏ `nums[r]` ở bên phải** (khi chi phí > `k`): phần tử cuối thuộc khối cuối `(b, c)`; chi phí giảm `b − nums[r]`; giảm `c` (bỏ khối nếu về 0).
- Cộng `r − l + 1` vào đáp án.

Mỗi phần tử vào/ra `deque` hằng số lần (khấu hao) ⇒ `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hai con trỏ + `cost(l, r)` bằng nhảy nhị phân trên chuỗi next greater

Max tiền tố của `[l, r]`: bắt đầu bằng `nums[l]` trên đoạn `[l, ng(l))`, rồi `nums[ng(l)]` trên `[ng(l), ng(ng(l)))`, … Gọi `up[e][i]` là chỉ số sau `2^e` lần nhảy, `W[e][i]` là tổng các khối đã đi qua. `Σ prefmax(l..r)` = tổng các khối **trọn vẹn** trước khối chứa `r` + phần còn lại. Trừ `Σ nums[l..r]` (tiền tố) ⇒ `cost(l, r)` trong `O(log n)`.

In [6]:
def count_lifting(nums, k):
    n = len(nums)
    ng = next_greater(nums)
    LOG = max(1, n.bit_length())
    up = [ng[:] + [n]]
    W = [[nums[i] * (ng[i] - i) for i in range(n)] + [0]]
    for e in range(1, LOG):
        pu, pw = up[-1], W[-1]
        up.append([pu[pu[i]] for i in range(n + 1)])
        W.append([pw[i] + pw[pu[i]] for i in range(n + 1)])
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)

    def cost(l, r):
        cur, acc = l, 0
        for e in range(LOG - 1, -1, -1):
            if up[e][cur] <= r:
                acc += W[e][cur]; cur = up[e][cur]
        acc += nums[cur] * (r - cur + 1)
        return acc - (pre[r + 1] - pre[l])

    ans, r = 0, 0
    for l in range(n):
        r = max(r, l)
        while r + 1 < n and cost(l, r + 1) <= k:
            r += 1
        ans += r - l + 1
    return ans

for nums, k, exp in examples:
    print(count_lifting(nums, k), '| kỳ vọng', exp)

17 | kỳ vọng 17
12 | kỳ vọng 12


### 7.2. Cách 3 — `deque` các khối max tiền tố, duyệt `l` từ phải sang (lời giải chính)

In [7]:
from collections import deque

def count_deque(nums, k):
    n = len(nums)
    dq = deque()               # các khối [giá trị, số phần tử], giá trị tăng dần từ trái sang
    cost, r, ans = 0, n - 1, 0
    for l in range(n - 1, -1, -1):
        v, cnt = nums[l], 1
        while dq and dq[0][0] <= v:
            b, c = dq.popleft()
            cost += (v - b) * c
            cnt += c
        dq.appendleft([v, cnt])
        while cost > k:
            b = dq[-1][0]
            cost -= b - nums[r]
            dq[-1][1] -= 1
            if dq[-1][1] == 0:
                dq.pop()
            r -= 1
        ans += r - l + 1
    return ans

for nums, k, exp in examples:
    print(count_deque(nums, k), '| kỳ vọng', exp)

17 | kỳ vọng 17
12 | kỳ vọng 12


### 7.3. Cách 4 — Nhị phân `R(l)` cho mỗi `l` với hàm `cost` của cách 2

Không dùng tính đơn điệu của `R(l)`: với mỗi `l`, nhị phân `r` lớn nhất có `cost(l, r) ≤ k` (vì `cost(l, ·)` không giảm). `O(n log² n)` — chậm hơn nhưng độc lập với lập luận hai con trỏ.

In [8]:
def count_bsearch(nums, k):
    n = len(nums)
    ng = next_greater(nums)
    LOG = max(1, n.bit_length())
    up = [ng[:] + [n]]
    W = [[nums[i] * (ng[i] - i) for i in range(n)] + [0]]
    for e in range(1, LOG):
        pu, pw = up[-1], W[-1]
        up.append([pu[pu[i]] for i in range(n + 1)])
        W.append([pw[i] + pw[pu[i]] for i in range(n + 1)])
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)
    def cost(l, r):
        cur, acc = l, 0
        for e in range(LOG - 1, -1, -1):
            if up[e][cur] <= r:
                acc += W[e][cur]; cur = up[e][cur]
        return acc + nums[cur] * (r - cur + 1) - (pre[r + 1] - pre[l])
    ans = 0
    for l in range(n):
        lo, hi = l, n - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if cost(l, mid) <= k: lo = mid
            else: hi = mid - 1
        ans += lo - l + 1
    return ans

for nums, k, exp in examples:
    print(count_bsearch(nums, k), '| kỳ vọng', exp)

17 | kỳ vọng 17
12 | kỳ vọng 12


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1.

In [9]:
def trace(nums, k):
    n = len(nums); dq = deque(); cost = 0; r = n - 1; ans = 0
    for l in range(n - 1, -1, -1):
        v, cnt = nums[l], 1
        merged = []
        while dq and dq[0][0] <= v:
            b, c = dq.popleft(); cost += (v - b) * c; cnt += c; merged.append((b, c))
        dq.appendleft([v, cnt])
        removed = []
        while cost > k:
            b = dq[-1][0]; cost -= b - nums[r]; dq[-1][1] -= 1
            if dq[-1][1] == 0: dq.pop()
            removed.append(r); r -= 1
        ans += r - l + 1
        print(f"l={l} (thêm {v}, nuốt {merged}) bỏ phải {removed} -> khối {[tuple(x) for x in dq]}, "
              f"chi phí {cost}, cửa sổ [{l},{r}] +{r - l + 1} (tổng {ans})")

trace(*examples[0][:2])

l=5 (thêm 4, nuốt []) bỏ phải [] -> khối [(4, 1)], chi phí 0, cửa sổ [5,5] +1 (tổng 1)
l=4 (thêm 4, nuốt [(4, 1)]) bỏ phải [] -> khối [(4, 2)], chi phí 0, cửa sổ [4,5] +2 (tổng 3)
l=3 (thêm 2, nuốt []) bỏ phải [] -> khối [(2, 1), (4, 2)], chi phí 0, cửa sổ [3,5] +3 (tổng 6)
l=2 (thêm 1, nuốt []) bỏ phải [] -> khối [(1, 1), (2, 1), (4, 2)], chi phí 0, cửa sổ [2,5] +4 (tổng 10)
l=1 (thêm 3, nuốt [(1, 1), (2, 1)]) bỏ phải [] -> khối [(3, 3), (4, 2)], chi phí 3, cửa sổ [1,5] +5 (tổng 15)
l=0 (thêm 6, nuốt [(3, 3), (4, 2)]) bỏ phải [5, 4, 3, 2] -> khối [(6, 2)], chi phí 3, cửa sổ [0,1] +2 (tổng 17)


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def countNonDecreasingSubarrays(self, nums: List[int], k: int) -> int:
        n = len(nums)
        dq = deque()
        cost = 0
        r = n - 1
        ans = 0
        for l in range(n - 1, -1, -1):
            v = nums[l]
            cnt = 1
            while dq and dq[0][0] <= v:
                b, c = dq.popleft()
                cost += (v - b) * c
                cnt += c
            dq.appendleft([v, cnt])
            while cost > k:
                last = dq[-1]
                cost -= last[0] - nums[r]
                last[1] -= 1
                if last[1] == 0:
                    dq.pop()
                r -= 1
            ans += r - l + 1
        return ans

for nums, k, exp in examples:
    got = Solution().countNonDecreasingSubarrays(nums, k)
    print(got, 'OK' if got == exp else 'SAI')

17 OK
12 OK


## 10. Vì sao thuật toán đúng

1. **Chi phí** đúng như 3.1 (cận dưới + cách đạt được).
2. **Bất biến của `deque`:** sau khi xử lý `l`, các khối (từ trái) mô tả chính xác dãy max tiền tố của `nums[l..r]`: giá trị khối tăng **thực sự**, tổng số phần tử = `r − l + 1`, và `cost = Σ (giá trị khối chứa j − nums[j])`.
   - Thêm `v` bên trái: mọi phần tử có max tiền tố `≤ v` giờ có max tiền tố `v` (tăng `v − b`); các khối có giá trị `> v` không đổi.
   - Bỏ phần tử phải nhất `r`: nó thuộc khối cuối; bỏ đóng góp `b − nums[r]`.
3. **Hai con trỏ:** `R(l)` không tăng khi `l` giảm (mục 3.2 theo chiều ngược lại), nên `r` chỉ giảm; khi `cost ≤ k` thì `[l, r]` là cửa sổ khả thi dài nhất bắt đầu tại `l`, đóng góp `r − l + 1` mảng con.
4. `r ≥ l` luôn đúng vì mảng con một phần tử có chi phí 0.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mở rộng từng `l` | `O(n²)` | `O(1)` |
| 2. Hai con trỏ + binary lifting | `O(n log n)` | `O(n log n)` |
| 3. `deque` khối | `O(n)` khấu hao | `O(n)` |
| 4. Nhị phân mỗi `l` | `O(n log² n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'lifting': count_lifting, 'deque': count_deque, 'bsearch': count_bsearch,
         'submit': lambda a, k: Solution().countNonDecreasingSubarrays(a, k)}
edge = [([5], 1, 1), ([1, 2, 3], 1, 6), ([3, 2, 1], 1, 5), ([3, 2, 1], 3, 6), ([10, 1, 10, 1], 8, 5)]
for nums, k, exp in examples + edge:
    assert count_brute(nums, k) == exp, (nums, k, count_brute(nums, k))
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3420)
cnt = 0
for _ in range(3000):
    n = rng.randint(1, 14)
    nums = [rng.randint(1, 6) for _ in range(n)]
    k = rng.randint(1, 12)
    exp = count_brute(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 3000 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [13]:
import time
n = 10 ** 5
cases = {
    'ngẫu nhiên, k = 10^9': ([rng.randint(1, 10 ** 9) for _ in range(n)], 10 ** 9),
    'giảm dần, k lớn': (list(range(n, 0, -1)), 10 ** 9),
    'răng cưa nhỏ, k = 10^5': ([rng.randint(1, 100) for _ in range(n)], 10 ** 5),
    'tăng dần': (list(range(1, n + 1)), 1),
}
for name, (nums, k) in cases.items():
    t = time.time(); a = Solution().countNonDecreasingSubarrays(nums, k); t1 = time.time() - t
    t = time.time(); b = count_lifting(nums, k); t2 = time.time() - t
    assert a == b
    print(f"{name:24s}: {a:>12d} | deque {t1:.2f}s | binary lifting {t2:.2f}s")

ngẫu nhiên, k = 10^9    :       476509 | deque 0.03s | binary lifting 0.33s


giảm dần, k lớn         :   3472138440 | deque 0.02s | binary lifting 0.26s


răng cưa nhỏ, k = 10^5  :    200417463 | deque 0.03s | binary lifting 0.29s


tăng dần                :   5000050000 | deque 0.02s | binary lifting 0.26s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chi phí = tổng các "bước giảm" giữa hai phần tử kề nhau

In [14]:
def wrong_adjacent(nums, k):
    n, c = len(nums), 0
    for l in range(n):
        cost = 0
        for r in range(l, n):
            if r > l:
                cost += max(0, nums[r - 1] - nums[r])     # SAI: phải so với max tiền tố
            if cost > k:
                break
            c += 1
    return c

print('chi phí [3,1,2]: sai =', max(0, 3 - 1) + max(0, 1 - 2), '| đúng =', cost_direct([3, 1, 2]))
print('sai :', wrong_adjacent([3, 1, 2], 2), '| đúng:', Solution().countNonDecreasingSubarrays([3, 1, 2], 2))

chi phí [3,1,2]: sai = 2 | đúng = 3
sai : 6 | đúng: 5


**Vì sao sai:** sau khi tăng `1 → 3`, phần tử `2` phải lên `3` chứ không chỉ ≥ `1`.

### Lỗi 2 — Hai con trỏ trái → phải, bỏ phần tử trái chỉ trừ đóng góp của chính nó

In [15]:
def wrong_left_removal(nums, k):
    n, ans, r = len(nums), 0, -1
    contrib = [0] * n                    # đóng góp lúc thêm (dựa trên max cửa sổ khi đó)
    cost, mx = 0, 0
    for l in range(n):
        if r < l:
            r, cost, mx = l, 0, nums[l]; contrib[l] = 0
        while r + 1 < n:
            nmx = max(mx, nums[r + 1])
            if cost + nmx - nums[r + 1] > k:
                break
            r += 1; mx = nmx; contrib[r] = mx - nums[r]; cost += contrib[r]
        ans += r - l + 1
        cost -= contrib[l]                    # SAI: các phần tử sau vẫn giữ max cũ (có thể do nums[l])
    return ans

rng = random.Random(2)
ce = next((a, k) for a, k in (([rng.randint(1, 6) for _ in range(rng.randint(2, 7))], rng.randint(1, 6)) for _ in range(20000))
          if wrong_left_removal(a, k) != count_brute(a, k))
print('phản ví dụ:', ce, '| sai:', wrong_left_removal(*ce), '| đúng:', count_brute(*ce))

phản ví dụ: ([2, 5, 1, 5, 6, 2], 4) | sai: 18 | đúng: 19


**Vì sao sai:** bỏ phần tử trái có thể **hạ** max tiền tố của nhiều phần tử phía sau; chi phí giảm nhiều hơn đóng góp của riêng nó. Vì vậy lời giải duyệt `l` từ **phải sang** (thêm bên trái chỉ làm max **tăng**, dễ cập nhật bằng các khối).

### Lỗi 3 — Khi gộp khối quên nhân với **số phần tử** của khối

In [16]:
def wrong_merge_count(nums, k):
    n = len(nums); dq = deque(); cost = 0; r = n - 1; ans = 0
    for l in range(n - 1, -1, -1):
        v, cnt = nums[l], 1
        while dq and dq[0][0] <= v:
            b, c = dq.popleft()
            cost += v - b                                # SAI: phải là (v - b) * c
            cnt += c
        dq.appendleft([v, cnt])
        while cost > k:
            cost -= dq[-1][0] - nums[r]
            dq[-1][1] -= 1
            if dq[-1][1] == 0: dq.pop()
            r -= 1
        ans += r - l + 1
    return ans

print('sai :', wrong_merge_count([6, 3, 1, 2, 4, 4], 7), '| đúng:', Solution().countNonDecreasingSubarrays([6, 3, 1, 2, 4, 4], 7))

sai : 20 | đúng: 17


**Vì sao sai:** mọi phần tử trong khối bị "nuốt" đều được nâng max tiền tố từ `b` lên `v` — chi phí tăng `(v − b)` **cho từng phần tử**. Thiếu hệ số `c` làm chi phí bị đánh giá thấp ⇒ đếm thừa mảng con.

### Lỗi 4 — Cộng `r − l` (quên chính mảng con một phần tử)

In [17]:
def wrong_off_by_one(nums, k):
    return Solution().countNonDecreasingSubarrays(nums, k) - len(nums)

print('sai :', wrong_off_by_one([6, 3, 1, 2, 4, 4], 7), '| đúng:', Solution().countNonDecreasingSubarrays([6, 3, 1, 2, 4, 4], 7))

sai : 11 | đúng: 17


**Vì sao sai:** cửa sổ `[l, r]` chứa `r − l + 1` mảng con bắt đầu tại `l` (kể cả `[l, l]`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chỉ được tăng để không giảm" ⇒ chi phí `Σ (max tiền tố − a_j)`.
- "Đếm mảng con thỏa điều kiện đơn điệu" ⇒ hai con trỏ; chọn **hướng duyệt** sao cho cập nhật dễ (ở đây thêm bên trái).
- Max tiền tố khi thêm phần tử ở đầu ⇒ ngăn xếp/`deque` đơn điệu các khối.
- Cần truy vấn `cost(l, r)` tùy ý ⇒ binary lifting trên next greater + tổng tiền tố.

## 15. Biến thể của bài

1. Chỉ được **giảm** để không giảm ⇒ đối xứng: min hậu tố, duyệt ngược chiều.
2. Làm mảng con **không tăng** bằng cách tăng ⇒ đảo mảng.
3. Đếm mảng con có chi phí **đúng bằng** `k` ⇒ `F(k) − F(k − 1)` với `F` = số mảng con chi phí ≤ `k`.
4. Bài liên quan: LC 2944, LC 1918; LC 2762 (Continuous Subarrays) — hai con trỏ + cấu trúc đơn điệu.

In [18]:
# Biến thể 3: đếm mảng con có chi phí ĐÚNG BẰNG k
def count_exact(nums, k):
    f = lambda kk: Solution().countNonDecreasingSubarrays(nums, kk) if kk >= 1 else \
        sum(1 for l in range(len(nums)) for r in range(l, len(nums)) if cost_direct(nums[l:r + 1]) <= kk)
    return f(k) - f(k - 1)

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 9))]; k = rng.randint(1, 8)
    exp = sum(1 for l in range(len(nums)) for r in range(l, len(nums)) if cost_direct(nums[l:r + 1]) == k)
    assert count_exact(nums, k) == exp
print('Biến thể "đúng bằng k" khớp vét cạn; ví dụ 1 với k = 1..8:', [count_exact([6, 3, 1, 2, 4, 4], kk) for kk in range(1, 9)])

Biến thể "đúng bằng k" khớp vét cạn; ví dụ 1 với k = 1..8: [0, 1, 4, 0, 0, 0, 0, 1]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mở rộng từng `l` | `O(n²)` | Oracle |
| 2 | Hai con trỏ + binary lifting | `O(n log n)` | `cost(l, r)` tùy ý |
| 3 | `deque` khối, duyệt ngược | `O(n)` | **Nộp bài** |
| 4 | Nhị phân mỗi `l` | `O(n log² n)` | Kiểm chứng độc lập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [1, 1, 1]`, `k = 1`.

**Bài 2 (tính tay):** `nums = [4, 1, 1]`, `k = 5`.

**Bài 3 (điền TODO):** viết `min_ops(nums)` — chi phí làm **cả mảng** không giảm.

**Bài 4 (tự viết lại):** biến thể 1 — chỉ được **giảm** phần tử.

**Bài 5 (mini challenge):** trả lời `q` truy vấn `cost(l, r)` bằng cách 2.

In [19]:
def min_ops_todo(nums):
    # TODO
    raise NotImplementedError

def count_decrease_only_todo(nums, k):
    # TODO
    raise NotImplementedError

def answer_cost_queries_todo(nums, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Mảng đã không giảm. (2) Mọi mảng con chi phí 0. (3) `6`.

**Bài 2.** (1) `[4,1,1]` tốn `3 + 3 = 6 > 5`. (2) Các mảng con khác? (3) Đếm: 3 đơn + `[4,1]` (3) + `[1,1]` (0) = 5.

**Bài 3.** (1) Chính là `cost_direct`. (2) Một lần quét. (3) `O(n)`.

**Bài 4.** (1) Chỉ giảm ⇒ phần tử `j` phải ≤ min hậu tố. (2) Chi phí `Σ (a_j − suffix_min)`. (3) Đảo mảng và đổi dấu để dùng lại lời giải.

**Bài 5.** (1) Dựng `up`, `W`. (2) Mỗi truy vấn `O(log n)`. (3) Đối chiếu `cost_direct`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().countNonDecreasingSubarrays([1, 1, 1], 1)
a2 = Solution().countNonDecreasingSubarrays([4, 1, 1], 5)
assert a1 == 6 and a2 == count_brute([4, 1, 1], 5) == 5
min_ops = cost_direct
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3:', min_ops([6, 3, 1, 2, 4, 4]))

# Bài 4: chỉ giảm. Giảm a_j để không giảm <=> đảo mảng + đổi dấu thành bài "chỉ tăng"
def count_decrease_only(nums, k):
    return Solution().countNonDecreasingSubarrays([-x + 10 ** 9 + 1 for x in reversed(nums)], k)

def brute_decrease(nums, k):
    n, c = len(nums), 0
    for l in range(n):
        for r in range(l, n):
            mn, cost = float('inf'), 0
            for x in reversed(nums[l:r + 1]):
                mn = min(mn, x); cost += x - mn
            c += cost <= k
    return c

rng = random.Random(4)
for _ in range(300):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 8))]; k = rng.randint(1, 8)
    assert count_decrease_only(nums, k) == brute_decrease(nums, k)
print('Bài 4 khớp vét cạn.')

Bài 1: 6 | Bài 2: 5 | Bài 3: 16
Bài 4 khớp vét cạn.


In [21]:
# Bài 5 — truy vấn cost(l, r)
def build_cost_oracle(nums):
    n = len(nums); ng = next_greater(nums); LOG = max(1, n.bit_length())
    up = [ng[:] + [n]]; W = [[nums[i] * (ng[i] - i) for i in range(n)] + [0]]
    for e in range(1, LOG):
        pu, pw = up[-1], W[-1]
        up.append([pu[pu[i]] for i in range(n + 1)]); W.append([pw[i] + pw[pu[i]] for i in range(n + 1)])
    pre = [0]
    for x in nums:
        pre.append(pre[-1] + x)
    def cost(l, r):
        cur, acc = l, 0
        for e in range(LOG - 1, -1, -1):
            if up[e][cur] <= r:
                acc += W[e][cur]; cur = up[e][cur]
        return acc + nums[cur] * (r - cur + 1) - (pre[r + 1] - pre[l])
    return cost

rng = random.Random(5)
nums = [rng.randint(1, 50) for _ in range(300)]
cost = build_cost_oracle(nums)
for _ in range(2000):
    l = rng.randrange(300); r = rng.randrange(l, 300)
    assert cost(l, r) == cost_direct(nums[l:r + 1])
print('Bài 5: 2000 truy vấn cost(l, r) khớp tính trực tiếp.')

Bài 5: 2000 truy vấn cost(l, r) khớp tính trực tiếp.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dq = deque()   # [giá trị, số phần tử], tăng dần từ trái
cost = 0; r = n-1; ans = 0
for l in range(n-1, -1, -1):
    gộp các khối đầu có giá trị <= nums[l] (cost += (nums[l]-b)*c), appendleft khối mới
    while cost > k: cost -= dq[-1][0] - nums[r]; giảm khối cuối; r -= 1
    ans += r - l + 1
```

**Checklist:**

- [ ] Chi phí theo **max tiền tố**, không theo phần tử kề?
- [ ] Duyệt `l` từ phải sang (thêm bên trái làm max chỉ tăng)?
- [ ] Bỏ bên phải dùng giá trị **khối cuối**?
- [ ] Cộng `r − l + 1`?
- [ ] Số lớn (`(v − b)·c` tới `10^{14}`) — C++/Java cần `long long`?